In [23]:
import torch
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\venuv\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\venuv\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [24]:
pdfFilePath = r"C:\Users\venuv\Documents\ai developer\public\attentionIsAllYouNeedPaper.pdf".replace("\\", "/")

In [38]:
#read pdf file and get text data from it
from pypdf import PdfReader
chunkSize = 550
overlap = 100 
chunks = []
ids = []
reader = PdfReader(pdfFilePath)
pendingChunk = ""
partitionNumber = 0
for pageNum, page in enumerate(reader.pages):
    pageText = page.extract_text()
    import re

    pageText = re.sub(
        r'-\n',
        '',
        pageText
    )

    pageText = pageText.replace('\n', ' ')

    pageTextLength = len(pageText)
    exploringPoint = 0
    
    if(len(pendingChunk)!=0):
        remainingLength = chunkSize-len(pendingChunk)
        remainingLength = min(remainingLength, len(pageText))
        pendingChunk+=pageText[:remainingLength]
        chunks.append(pendingChunk)
        pendingChunk = ""
        ids.append(str(pageNum-1)+"-"+str(partitionNumber))
        exploringPoint = remainingLength-max(0, exploringPoint-overlap)

    partitionNumber = 0
    for i in range(exploringPoint,pageTextLength-chunkSize, chunkSize-overlap):
        exploringPoint = i+chunkSize
        chunk = pageText[i:exploringPoint]
        chunks.append(chunk)
        ids.append(str(pageNum)+"-"+str(partitionNumber))
        partitionNumber+=1

    exploringPoint = min(exploringPoint, max(0, pageTextLength-overlap))

    pendingChunk += pageText[exploringPoint:]

if pendingChunk:
    chunks.append(pendingChunk)
    ids.append(str(len(reader.pages)-1)+"-"+str(partitionNumber))
    

In [39]:
#store chunks in db
import chromadb
client = chromadb.Client()

In [41]:
collection = client.create_collection(
    name="pdf"
)


In [42]:
for chunk, chunkId in zip(chunks, ids):
    collection.add(
        documents=[chunk],
        ids=[chunkId],
        metadatas = [
            {
                "file":"attentionIsAllYouNeedPaper.pdf",
                "pagePart":chunkId
            }
        ]
    )

In [43]:
Question = "What is self attention?"

In [44]:
results = collection.query(
    query_texts = [Question],
    n_results = 3
)

In [45]:
results

{'ids': [['1-4', '6-1', '4-1']],
 'embeddings': None,
 'documents': [['effect we counteract with Multi-Head Attention as described in section 3.2. Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [4, 22, 23, 19]. End-to-end memory networks are based on a recurrent attention mechanism instead ',
   ' Even with k = n, however, the complexity of a separable convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer, the approach we take in our model. As side beneﬁt, self-attention could yield more interpretable models. We inspect attention distributions from our models and present and discuss examples in the appendix. Not on

In [46]:
context = "" 
for doc in results["documents"][0]:
    context+=doc+"\n"


In [47]:
prompt = f"""
Context: {context}

Question:{Question}

Answer the question from the context
"""

In [48]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
modelName = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(
    modelName
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    modelName
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

output = model.generate(
    **inputs,
    max_new_tokens=500,
    temperature=0.7,
    do_sample=True
)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 4152.51it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [49]:
answer = tokenizer.decode(
    output[0]
)

In [50]:
answer

'<pad> an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence</s>'